# Movie Recommendation System

This notebook builds a movie recommendation system using three approaches:

1. **Content-Based Filtering** — recommends movies similar in genre/content to ones a user liked, using TF-IDF and cosine similarity.
2. **Collaborative Filtering** — recommends movies based on patterns in user rating behavior, using matrix factorization (SVD) via `TruncatedSVD` and also an item-based k-NN approach.
3. **Hybrid Recommender** — combines both approaches into a single weighted score.

We evaluate the system with standard metrics (RMSE for rating prediction, Precision@K / Recall@K for top-N recommendation quality), and demo recommendations for sample users.

**Dataset:** A synthetic MovieLens-style dataset (movies with genres + a user-item ratings matrix) generated within this notebook, so it runs fully offline with no external downloads. The generation process mimics real-world rating behavior (user preference profiles, popularity bias, sparsity) so the modeling techniques below transfer directly to real datasets like MovieLens 100K/1M.


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
RNG = np.random.default_rng(42)

pd.set_option("display.max_columns", None)


## 2. Generate the Dataset

We create:
- `movies`: movie_id, title, year, and a set of genres per movie (multi-label, like real MovieLens data)
- `ratings`: user_id, movie_id, rating (1-5), timestamp

To make the synthetic ratings realistic, each **user** gets a latent genre-preference vector, and each **movie** gets a latent genre vector (derived from its genres) plus a popularity/quality bias. A user's rating for a movie is a noisy function of the dot product of their preference vector and the movie's vector, plus the movie's baseline quality. This creates genuine, learnable structure — similar to real-world rating data — instead of pure noise.

In [ ]:
GENRES = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Drama",
          "Fantasy", "Horror", "Mystery", "Romance", "Sci-Fi", "Thriller"]

N_MOVIES = 200
N_USERS = 300

title_adjectives = ["Silent", "Last", "Broken", "Hidden", "Golden", "Midnight", "Lost",
                     "Eternal", "Crimson", "Distant", "Forgotten", "Wild", "Secret", "Frozen"]
title_nouns = ["Horizon", "Kingdom", "Shadow", "Journey", "Legacy", "Empire", "Storm",
               "Garden", "City", "River", "Star", "Voyage", "Dream", "Echo"]

def make_title(i):
    adj = title_adjectives[RNG.integers(0, len(title_adjectives))]
    noun = title_nouns[RNG.integers(0, len(title_nouns))]
    return f"{adj} {noun}"

movie_rows = []
for mid in range(1, N_MOVIES + 1):
    n_genres = RNG.integers(1, 4)
    genres = RNG.choice(GENRES, size=n_genres, replace=False)
    year = int(RNG.integers(1980, 2025))
    quality = RNG.normal(0, 1)  # latent baseline quality/popularity bias
    movie_rows.append({
        "movie_id": mid,
        "title": f"{make_title(mid)} ({year})",
        "year": year,
        "genres": "|".join(genres),
        "_quality": quality
    })

movies = pd.DataFrame(movie_rows)

# Movie latent vector = normalized genre indicator (used to generate ratings, not shown to models)
genre_matrix = np.zeros((N_MOVIES, len(GENRES)))
for idx, row in movies.iterrows():
    for g in row["genres"].split("|"):
        genre_matrix[idx, GENRES.index(g)] = 1

# Each user has a genre preference vector (how much they like each genre)
user_prefs = RNG.normal(0, 1, size=(N_USERS, len(GENRES)))
user_strictness = RNG.uniform(0.5, 1.5, size=N_USERS)  # some users rate more harshly/generously

# Simulate sparse ratings: each user rates a random subset of movies (biased toward genres they like)
rating_rows = []
for uid in range(1, N_USERS + 1):
    u_vec = user_prefs[uid - 1]
    affinity = genre_matrix @ u_vec  # affinity per movie
    # probability of having watched/rated a movie increases with affinity + popularity
    watch_prob = 1 / (1 + np.exp(-(affinity * 0.5 + movies["_quality"].values * 0.3)))
    watch_prob = watch_prob / watch_prob.sum()
    n_ratings = int(RNG.integers(15, 60))
    rated_movies = RNG.choice(movies["movie_id"].values, size=n_ratings, replace=False, p=watch_prob)

    for mid in rated_movies:
        midx = mid - 1
        raw_score = (affinity[midx] * 0.6 + movies.loc[midx, "_quality"] * 0.8) * user_strictness[uid - 1]
        noise = RNG.normal(0, 0.6)
        rating = raw_score + noise
        # squash to 1-5 scale
        rating = 3 + rating
        rating = np.clip(rating, 1, 5)
        rating = round(rating * 2) / 2  # allow half-star ratings
        rating_rows.append({"user_id": uid, "movie_id": mid, "rating": rating})

ratings = pd.DataFrame(rating_rows)
movies = movies.drop(columns=["_quality"])  # hide latent variable from downstream modeling

print(f"Movies: {movies.shape[0]}")
print(f"Ratings: {ratings.shape[0]}")
print(f"Users: {ratings['user_id'].nunique()}")
print(f"Sparsity: {1 - len(ratings) / (N_USERS * N_MOVIES):.2%}")
movies.head()


In [ ]:
ratings.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ratings["rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Ratings")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")

genre_counts = movies["genres"].str.split("|").explode().value_counts()
genre_counts.plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Movie Count by Genre")
axes[1].set_xlabel("Genre")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
ratings_per_user = ratings.groupby("user_id").size()
ratings_per_movie = ratings.groupby("movie_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(ratings_per_user, bins=20, color="teal")
axes[0].set_title("Ratings per User")
axes[0].set_xlabel("# Ratings")

axes[1].hist(ratings_per_movie, bins=20, color="darkorange")
axes[1].set_title("Ratings per Movie")
axes[1].set_xlabel("# Ratings")

plt.tight_layout()
plt.show()

print(f"Avg ratings/user: {ratings_per_user.mean():.1f}")
print(f"Avg ratings/movie: {ratings_per_movie.mean():.1f}")


## 4. Train / Test Split

We hold out 20% of each user's ratings as a test set. This lets us evaluate rating-prediction accuracy (RMSE) and ranking quality (Precision@K) on unseen interactions.

In [ ]:
train_df, test_df = train_test_split(
    ratings, test_size=0.2, random_state=42, stratify=ratings["user_id"]
)
print(f"Train: {len(train_df)}  Test: {len(test_df)}")

user_item_train = train_df.pivot_table(index="user_id", columns="movie_id", values="rating")
user_ids = user_item_train.index.tolist()
movie_ids_in_train = user_item_train.columns.tolist()
print(f"User-item matrix shape: {user_item_train.shape}")


## 5. Content-Based Filtering

We build a TF-IDF representation of each movie's genres, then use cosine similarity between movies to recommend titles similar to ones a user has rated highly.

In [ ]:
# Represent genres as a "document" of space-separated tokens for TF-IDF
movies["genre_soup"] = movies["genres"].str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies["genre_soup"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

content_sim = linear_kernel(tfidf_matrix, tfidf_matrix)  # cosine similarity since TF-IDF rows are L2-normalized
content_sim_df = pd.DataFrame(content_sim, index=movies["movie_id"], columns=movies["movie_id"])


In [ ]:
def content_based_recommend(user_id, n=10, ratings_source=train_df):
    """Recommend movies similar to ones the user rated highly (>=4), excluding already-rated movies."""
    user_ratings = ratings_source[ratings_source["user_id"] == user_id]
    liked = user_ratings[user_ratings["rating"] >= 4]["movie_id"].tolist()
    if not liked:
        # fallback: use user's highest-rated movies regardless of threshold
        liked = user_ratings.sort_values("rating", ascending=False)["movie_id"].head(3).tolist()
    if not liked:
        return pd.DataFrame(columns=["movie_id", "title", "score"])

    sim_scores = content_sim_df.loc[liked].mean(axis=0)
    already_rated = set(user_ratings["movie_id"])
    sim_scores = sim_scores.drop(labels=[m for m in already_rated if m in sim_scores.index], errors="ignore")

    top = sim_scores.sort_values(ascending=False).head(n)
    result = movies[movies["movie_id"].isin(top.index)].copy()
    result["score"] = result["movie_id"].map(top)
    return result[["movie_id", "title", "genres", "score"]].sort_values("score", ascending=False)

# Demo
demo_user = user_ids[0]
print(f"Content-based recommendations for user {demo_user}:")
content_based_recommend(demo_user, n=10)


## 6. Collaborative Filtering

### 6a. Matrix Factorization (SVD)

We fill missing entries with each user's mean rating, then decompose the matrix with `TruncatedSVD` to learn latent factors, and reconstruct predicted ratings.

In [ ]:
user_means = user_item_train.mean(axis=1)
user_item_filled = user_item_train.sub(user_means, axis=0).fillna(0)

N_FACTORS = 20
svd = TruncatedSVD(n_components=N_FACTORS, random_state=42)
user_factors = svd.fit_transform(user_item_filled)
item_factors = svd.components_

pred_matrix = user_factors @ item_factors
pred_df = pd.DataFrame(pred_matrix, index=user_item_train.index, columns=user_item_train.columns)
pred_df = pred_df.add(user_means, axis=0)
pred_df = pred_df.clip(1, 5)

print("Explained variance ratio (sum):", svd.explained_variance_ratio_.sum().round(3))


In [ ]:
def cf_predict(user_id, movie_id):
    if user_id not in pred_df.index or movie_id not in pred_df.columns:
        return train_df["rating"].mean()  # cold-start fallback
    return pred_df.loc[user_id, movie_id]

def cf_recommend(user_id, n=10, ratings_source=train_df):
    if user_id not in pred_df.index:
        return pd.DataFrame(columns=["movie_id", "title", "score"])
    already_rated = set(ratings_source[ratings_source["user_id"] == user_id]["movie_id"])
    scores = pred_df.loc[user_id].drop(labels=[m for m in already_rated if m in pred_df.columns], errors="ignore")
    top = scores.sort_values(ascending=False).head(n)
    result = movies[movies["movie_id"].isin(top.index)].copy()
    result["score"] = result["movie_id"].map(top)
    return result[["movie_id", "title", "genres", "score"]].sort_values("score", ascending=False)

print(f"Collaborative-filtering recommendations for user {demo_user}:")
cf_recommend(demo_user, n=10)


### 6b. Item-Based k-NN (alternative CF approach)

In [ ]:
item_sim = cosine_similarity(user_item_filled.T)
item_sim_df = pd.DataFrame(item_sim, index=user_item_train.columns, columns=user_item_train.columns)

def item_knn_recommend(user_id, n=10, k=15, ratings_source=train_df):
    user_ratings = ratings_source[ratings_source["user_id"] == user_id].set_index("movie_id")["rating"]
    if user_ratings.empty:
        return pd.DataFrame(columns=["movie_id", "title", "score"])

    scores = {}
    for candidate in user_item_train.columns:
        if candidate in user_ratings.index:
            continue
        sims = item_sim_df.loc[candidate, user_ratings.index]
        top_k = sims.sort_values(ascending=False).head(k)
        if top_k.sum() == 0:
            continue
        weighted = (top_k.values * user_ratings.loc[top_k.index].values).sum() / top_k.sum()
        scores[candidate] = weighted

    top = pd.Series(scores).sort_values(ascending=False).head(n)
    result = movies[movies["movie_id"].isin(top.index)].copy()
    result["score"] = result["movie_id"].map(top)
    return result[["movie_id", "title", "genres", "score"]].sort_values("score", ascending=False)

print(f"Item-based k-NN recommendations for user {demo_user}:")
item_knn_recommend(demo_user, n=10)


## 7. Hybrid Recommender

We combine content-based and collaborative-filtering scores into a single weighted ranking. Both score sets are min-max normalized to [0, 1] before blending so neither approach dominates due to scale differences.

In [ ]:
def normalize(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

def hybrid_recommend(user_id, n=10, alpha=0.5, ratings_source=train_df):
    """alpha: weight on collaborative filtering; (1-alpha) weight on content-based."""
    user_ratings = ratings_source[ratings_source["user_id"] == user_id]
    already_rated = set(user_ratings["movie_id"])

    # Content-based scores over full candidate set
    liked = user_ratings[user_ratings["rating"] >= 4]["movie_id"].tolist()
    if not liked:
        liked = user_ratings.sort_values("rating", ascending=False)["movie_id"].head(3).tolist()
    content_scores = content_sim_df.loc[liked].mean(axis=0) if liked else pd.Series(dtype=float)

    # CF scores
    cf_scores = pred_df.loc[user_id] if user_id in pred_df.index else pd.Series(dtype=float)

    candidates = set(movies["movie_id"]) - already_rated
    content_norm = normalize(content_scores.reindex(list(candidates)).fillna(0))
    cf_norm = normalize(cf_scores.reindex(list(candidates)).fillna(cf_scores.mean() if len(cf_scores) else 0))

    hybrid_score = alpha * cf_norm + (1 - alpha) * content_norm
    top = hybrid_score.sort_values(ascending=False).head(n)

    result = movies[movies["movie_id"].isin(top.index)].copy()
    result["score"] = result["movie_id"].map(top)
    return result[["movie_id", "title", "genres", "score"]].sort_values("score", ascending=False)

print(f"Hybrid recommendations for user {demo_user}:")
hybrid_recommend(demo_user, n=10, alpha=0.5)


## 8. Evaluation

**RMSE**: how close predicted ratings are to actual held-out ratings (evaluates the CF rating-prediction model).

**Precision@K / Recall@K**: of the top-K movies we recommend, how many did the user actually rate highly (>=4) in the held-out test set? This evaluates ranking quality, which is what actually matters for a recommender system.

In [ ]:
# --- RMSE on test set (collaborative filtering) ---
test_eval = test_df[test_df["user_id"].isin(pred_df.index) & test_df["movie_id"].isin(pred_df.columns)]
y_true = test_eval["rating"].values
y_pred = test_eval.apply(lambda r: cf_predict(r["user_id"], r["movie_id"]), axis=1).values

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"Collaborative Filtering RMSE on test set: {rmse:.3f}  (evaluated on {len(test_eval)} ratings)")

# Baseline: predict the global mean rating for every test point
baseline_pred = np.full_like(y_true, train_df["rating"].mean())
baseline_rmse = np.sqrt(mean_squared_error(y_true, baseline_pred))
print(f"Baseline (global mean) RMSE: {baseline_rmse:.3f}")


In [ ]:
def precision_recall_at_k(recommend_fn, k=10, rating_threshold=4, sample_users=None, **kwargs):
    """Average Precision@K and Recall@K across users, using test_df as ground truth."""
    if sample_users is None:
        sample_users = user_ids
    precisions, recalls = [], []

    for uid in sample_users:
        relevant = set(test_df[(test_df["user_id"] == uid) & (test_df["rating"] >= rating_threshold)]["movie_id"])
        if not relevant:
            continue
        recs = recommend_fn(uid, n=k, **kwargs)
        if recs.empty:
            continue
        recommended_ids = set(recs["movie_id"])
        hits = recommended_ids & relevant

        precisions.append(len(hits) / k)
        recalls.append(len(hits) / len(relevant))

    return np.mean(precisions), np.mean(recalls)

# Evaluate on a sample of users for speed
sample = user_ids[:80]

results = {}
for name, fn in [
    ("Content-Based", content_based_recommend),
    ("Collaborative (SVD)", cf_recommend),
    ("Item-Based kNN", item_knn_recommend),
    ("Hybrid", hybrid_recommend),
]:
    p, r = precision_recall_at_k(fn, k=10, sample_users=sample)
    results[name] = {"Precision@10": p, "Recall@10": r}
    print(f"{name:22s}  Precision@10={p:.3f}  Recall@10={r:.3f}")

results_df = pd.DataFrame(results).T
results_df


In [ ]:
results_df.plot(kind="bar", figsize=(9, 5), color=["steelblue", "indianred"])
plt.title("Recommender Comparison: Precision@10 vs Recall@10")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 9. Summary & Discussion

- **Content-based filtering** recommends movies with similar genre profiles to what a user already liked. It works even for movies with very few ratings ("item cold-start"), but tends to recommend overly similar items and can't capture taste signals beyond the content features we gave it (here, just genre).
- **Collaborative filtering (SVD)** learns latent taste dimensions purely from the rating patterns, capturing more nuanced preferences (e.g. that a user likes a *specific style* of Sci-Fi, not just the Sci-Fi label). It struggles with cold-start users/items that have few or no ratings.
- **Item-based k-NN** is a simpler, more interpretable collaborative approach — "users who liked what you liked also liked these movies" — and serves as a useful comparison point to SVD.
- **The hybrid model** blends both signals, which tends to be more robust: content-based scores provide a reasonable fallback when collaborative signal is sparse for a given user, while collaborative filtering captures deeper taste patterns that content features alone miss.

### Possible extensions
- Incorporate richer content features (plot text/overview via TF-IDF, cast, director) instead of genres alone.
- Try `surprise` library algorithms (SVD++, KNNBaseline) or deep learning approaches (neural collaborative filtering, two-tower models).
- Tune the hybrid blending weight `alpha` per user (e.g. weight collaborative filtering more heavily for users with many ratings, content-based more for new users).
- Add temporal dynamics (recent ratings weighted more heavily; trending movies).
- Replace the synthetic dataset with real MovieLens data (100K/1M/25M) for production-quality benchmarks — all code above is written to be dataset-agnostic and should work unchanged as long as `movies` (movie_id, title, genres) and `ratings` (user_id, movie_id, rating) dataframes are provided in the same schema.
